In [1]:
import torch
import os
import gc
import numpy as np
from ultralytics import YOLO
import tensorrt as trt

# --- Configuration Parameters ---
MODEL_NAME = 'yolov8m'  # Choose your YOLOv8 model (n, s, m, l, x)
PT_MODEL_PATH = r'C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_original_bs2048.pt' # Path to save/load the PyTorch model

IMAGE_SIZE_H = 960
IMAGE_SIZE_W = 960
CHANNELS = 3

# For ONNX export (use a small batch for the export trace itself)
ONNX_EXPORT_BATCH_SIZE = 1
ONNX_MODEL_FILENAME = f'{MODEL_NAME}_bs{ONNX_EXPORT_BATCH_SIZE}_img{IMAGE_SIZE_H}x{IMAGE_SIZE_W}_dynamic.onnx'
ONNX_OPSET = 12 # Recommended for YOLOv8

# For TensorRT Engine building
TARGET_MAX_INFERENCE_BATCH_SIZE = 512 # Your desired ultimate batch size
# Opt batch size should be representative, but can be smaller than max to save build VRAM
# If builder OOMs, try making this even smaller (e.g., 1 or a small power of 2)
TRT_OPT_BATCH_SIZE = max(1, TARGET_MAX_INFERENCE_BATCH_SIZE // 8) # e.g., 128 for 1024 max
TRT_MIN_BATCH_SIZE = 1
TRT_ENGINE_FILENAME = f'{MODEL_NAME}_bs{TRT_MIN_BATCH_SIZE}-{TRT_OPT_BATCH_SIZE}-{TARGET_MAX_INFERENCE_BATCH_SIZE}_img{IMAGE_SIZE_H}x{IMAGE_SIZE_W}_fp16.engine'
TRT_WORKSPACE_GB = 12  # Workspace in GB. Adjust based on your VRAM. Try higher if you have it, lower if OOM.

# --- Check CUDA Availability ---
if not torch.cuda.is_available():
    print("ERROR: CUDA is not available. TensorRT requires a NVIDIA GPU.")
    # exit() # Uncomment to stop if no CUDA
else:
    print(f"CUDA is available. GPU: {torch.cuda.get_device_name(0)}")

# Create an instance of the model, it will download if not present
print(f"\nLoading/Downloading PyTorch model: {MODEL_NAME}")
try:
    yolo_model = YOLO(PT_MODEL_PATH if os.path.exists(PT_MODEL_PATH) else f'{MODEL_NAME}.pt')
    if not os.path.exists(PT_MODEL_PATH):
         yolo_model.save(PT_MODEL_PATH) # Ensure it's saved if downloaded
    print("PyTorch model loaded successfully.")
except Exception as e:
    print(f"Error loading YOLO model: {e}")
    # exit() # Uncomment to stop if model loading fails

CUDA is available. GPU: NVIDIA GeForce RTX 3090

Loading/Downloading PyTorch model: yolov8m
PyTorch model loaded successfully.


In [2]:
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# Cell 3: Step 1 - Export to ONNX with Dynamic Batch Axis
# We use batch_size=1 for the ONNX export trace itself to minimize VRAM during this step.
# `dynamic=True` ensures the ONNX model has a dynamic batch dimension.
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
print(f"\n--- Step 1: Exporting {MODEL_NAME} to ONNX ---")
print(f"Target ONNX image size: {IMAGE_SIZE_H}x{IMAGE_SIZE_W}")
print(f"ONNX export trace batch size: {ONNX_EXPORT_BATCH_SIZE}")
print(f"ONNX dynamic axes: True (for batch)")
print(f"ONNX Opset: {ONNX_OPSET}")
print(f"Output ONNX file: {ONNX_MODEL_FILENAME}")

try:
    yolo_model.export(
        format='onnx',
        imgsz=[IMAGE_SIZE_H, IMAGE_SIZE_W],
        batch=ONNX_EXPORT_BATCH_SIZE, # Dummy batch for tracing
        dynamic=True,              # Creates dynamic batch axis (and potentially H,W if not fixed by imgsz)
        opset=ONNX_OPSET,
        half=True,               # Export ONNX weights in FP16 (can make this step lighter)
                                   # If False, weights are FP32 in ONNX, TRT can still convert to FP16 engine
        simplify=True              # Optional: uses onnx-simplifier
    )
    # The export function saves the model with a name like `yolov8n.onnx` by default
    # We will rename it for clarity if it exists with the default name
    default_onnx_name = f"{MODEL_NAME}.onnx"
    if os.path.exists(default_onnx_name) and not os.path.exists(ONNX_MODEL_FILENAME):
        os.rename(default_onnx_name, ONNX_MODEL_FILENAME)
        print(f"ONNX model exported and saved as: {ONNX_MODEL_FILENAME}")
    elif os.path.exists(ONNX_MODEL_FILENAME):
         print(f"ONNX model already exists or was saved as: {ONNX_MODEL_FILENAME}")
    else:
        print(f"ERROR: ONNX model export did not produce expected file {default_onnx_name} or {ONNX_MODEL_FILENAME}")


except Exception as e:
    print(f"ERROR during ONNX export: {e}")
    print("This might be due to VRAM limitations even at batch=1 for large models/imgsz.")
    print("If 'half=True' was used and failed, try 'half=False'.")

# Clear PyTorch model from memory (optional, but good practice before TensorRT build)
del yolo_model
gc.collect()
torch.cuda.empty_cache()
print("Cleaned up PyTorch model from memory.")


--- Step 1: Exporting yolov8m to ONNX ---
Target ONNX image size: 960x960
ONNX export trace batch size: 1
ONNX dynamic axes: True (for batch)
ONNX Opset: 12
Output ONNX file: yolov8m_bs1_img960x960_dynamic.onnx
Ultralytics 8.3.129  Python-3.10.16 torch-2.7.0+cu126 CPU (13th Gen Intel Core(TM) i7-13700K)
WARNING half=True only compatible with GPU export, i.e. use device=0


YOLOv8m summary (fused): 92 layers, 25,886,080 parameters, 0 gradients, 78.9 GFLOPs

PyTorch: starting from 'C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_original_bs2048.pt' with input shape (1, 3, 960, 960) BCHW and output shape(s) (1, 84, 18900) (49.7 MB)

ONNX: starting export with onnx 1.17.0 opset 12...
ONNX: slimming with onnxslim 0.1.53...
ONNX: export success  4.5s, saved as 'C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_original_bs2048.onnx' (98.9 MB)

Export complete (5.3s)
Results saved to C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights
Predict:         yolo predict task=detect model=C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_original_bs2048.onnx imgsz=960  
Validate:        yolo val task=detect model=C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_origin

In [3]:
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# Cell 4: (Optional) Verify ONNX Model with ONNX Runtime
# This step helps confirm the ONNX model is valid and has dynamic batch.
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
import onnxruntime
import numpy as np

print(f"\n--- Optional Step: Verifying ONNX Model ({ONNX_MODEL_FILENAME}) ---")

if not os.path.exists(ONNX_MODEL_FILENAME):
    print(f"ONNX model {ONNX_MODEL_FILENAME} not found. Skipping verification.")
else:
    try:
        ort_session = onnxruntime.InferenceSession(ONNX_MODEL_FILENAME, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        print("ONNX Runtime session created successfully.")

        # Get input details
        input_meta = ort_session.get_inputs()[0]
        input_name = input_meta.name
        input_shape_ort = input_meta.shape # e.g., ['batch_size', 3, 960, 960]
        print(f"ONNX Input Name: {input_name}")
        print(f"ONNX Input Shape (from metadata): {input_shape_ort}")

        # Check if batch is dynamic (usually a string like 'batch_size' or None)
        if isinstance(input_shape_ort[0], str) or input_shape_ort[0] is None:
            print("Batch dimension appears dynamic.")
        else:
            print(f"WARNING: Batch dimension appears fixed to {input_shape_ort[0]}. Expected dynamic.")

        # Test with a small batch
        test_batch_size = 2
        dummy_input = np.random.randn(test_batch_size, CHANNELS, IMAGE_SIZE_H, IMAGE_SIZE_W).astype(np.float32)
        
        print(f"Running inference with ONNX Runtime on a dummy batch of size {test_batch_size}...")
        ort_inputs = {input_name: dummy_input}
        ort_outs = ort_session.run(None, ort_inputs)

        print(f"ONNX Runtime inference successful. Number of outputs: {len(ort_outs)}")
        # For YOLO, output shape is typically (batch_size, num_detections_plus_class_scores, num_boxes)
        # e.g., (batch_size, 84, 8400) for COCO classes and default anchors
        # or (batch_size, num_classes + 4, num_predictions) e.g. (bs, 84, 8400) for YOLOv8 P5 models
        # The exact shape can vary. The important part is that ort_outs[0].shape[0] == test_batch_size
        print(f"Shape of first output: {ort_outs[0].shape}")
        if ort_outs[0].shape[0] == test_batch_size:
            print("Output batch size matches input batch size. Dynamic batch appears to be working.")
        else:
            print("WARNING: Output batch size does NOT match input. Check ONNX export.")
        del ort_session, ort_inputs, ort_outs, dummy_input
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"ERROR during ONNX model verification: {e}")


--- Optional Step: Verifying ONNX Model (yolov8m_bs1_img960x960_dynamic.onnx) ---
ONNX Runtime session created successfully.
ONNX Input Name: images
ONNX Input Shape (from metadata): ['batch', 3, 'height', 'width']
Batch dimension appears dynamic.
Running inference with ONNX Runtime on a dummy batch of size 2...
ONNX Runtime inference successful. Number of outputs: 1
Shape of first output: (2, 84, 18900)
Output batch size matches input batch size. Dynamic batch appears to be working.


In [7]:
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# Cell 5: Step 2 - Build TensorRT Engine from ONNX
# This is the VRAM-intensive step.
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
TRT_OPT_BATCH_SIZE = 1
TARGET_MAX_INFERENCE_BATCH_SIZE = 256
print(f"\n--- Step 2: Building TensorRT Engine ---")
print(f"ONNX Input Model: {ONNX_MODEL_FILENAME}")
print(f"Target Engine File: {TRT_ENGINE_FILENAME}")
print(f"Min Batch Size: {TRT_MIN_BATCH_SIZE}")
print(f"Opt Batch Size: {TRT_OPT_BATCH_SIZE}")
print(f"Max Batch Size: {TARGET_MAX_INFERENCE_BATCH_SIZE}")
print(f"Image Size: {IMAGE_SIZE_H}x{IMAGE_SIZE_W}")
print(f"Precision: FP16")
print(f"Workspace: {TRT_WORKSPACE_GB} GB")
print("This may take a VERY long time and require substantial VRAM...")

if not os.path.exists(ONNX_MODEL_FILENAME):
    print(f"ERROR: ONNX model {ONNX_MODEL_FILENAME} not found. Cannot build engine.")
    # exit() # Or raise an error

TRT_LOGGER = trt.Logger(trt.Logger.WARNING) # Or trt.Logger.INFO for more verbosity

builder = trt.Builder(TRT_LOGGER)
# For TRT 10, EXPLICIT_BATCH is the default and only mode.
# Explicitly setting it doesn't hurt and maintains a degree of backward compatibility.
network_flags = (1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
network = builder.create_network(network_flags)
config = builder.create_builder_config()
parser = trt.OnnxParser(network, TRT_LOGGER)

# --- Configure Builder ---
# Workspace (correct for TRT 8.4+ including TRT 10)
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, TRT_WORKSPACE_GB * (1 << 30))

# Precision
if builder.platform_has_fast_fp16:
    config.set_flag(trt.BuilderFlag.FP16)
    print("FP16 mode enabled.")
else:
    print("FP16 mode not supported or not fast on this platform. Building in FP32 (higher VRAM).")
# Add INT8 for further VRAM saving if needed (requires calibration)
# if builder.platform_has_fast_int8:
#     config.set_flag(trt.BuilderFlag.INT8)
#     # You would need to provide an INT8 calibrator instance here:
#     # config.int8_calibrator = YourINT8Calibrator(...)

# --- Parse ONNX Model ---
with open(ONNX_MODEL_FILENAME, "rb") as model_file:
    if not parser.parse(model_file.read()):
        print("ERROR: Failed to parse the ONNX file.")
        for error_idx in range(parser.num_errors):
            print(parser.get_error(error_idx))
        # exit() # Or raise an error
    print("ONNX model parsed successfully.")

# --- Define Optimization Profile ---
# The input tensor name is typically 'images' for YOLO models exported from Ultralytics
# If unsure, you can inspect network.get_input(0).name after parsing
input_tensor_name = network.get_input(0).name # Assuming the first input is the main image input
print(f"Network Input Tensor Name used for profile: {input_tensor_name}")

profile = builder.create_optimization_profile()
profile.set_shape(
    input_tensor_name,
    min=(TRT_MIN_BATCH_SIZE, CHANNELS, IMAGE_SIZE_H, IMAGE_SIZE_W),
    opt=(TRT_OPT_BATCH_SIZE, CHANNELS, IMAGE_SIZE_H, IMAGE_SIZE_W),
    max=(TARGET_MAX_INFERENCE_BATCH_SIZE, CHANNELS, IMAGE_SIZE_H, IMAGE_SIZE_W)
)
# If your model has other dynamic input shapes (e.g., sequence length), define them here too.
# For example: profile.set_shape("another_input", min_shape_another, opt_shape_another, max_shape_another)

config.add_optimization_profile(profile)
print(f"Optimization profile added for input '{input_tensor_name}':")
print(f"  Min Shape: {(TRT_MIN_BATCH_SIZE, CHANNELS, IMAGE_SIZE_H, IMAGE_SIZE_W)}")
print(f"  Opt Shape: {(TRT_OPT_BATCH_SIZE, CHANNELS, IMAGE_SIZE_H, IMAGE_SIZE_W)}")
print(f"  Max Shape: {(TARGET_MAX_INFERENCE_BATCH_SIZE, CHANNELS, IMAGE_SIZE_H, IMAGE_SIZE_W)}")


# --- Build Engine ---
print("Building serialized network (engine)...")
# This try-except might not catch all CUDA OOM errors gracefully as they can be low-level
try:
    serialized_engine = builder.build_serialized_network(network, config)

    if serialized_engine is None:
        print("ERROR: Failed to build the TensorRT engine. This is often due to OOM or configuration issues.")
        print("Try the following:")
        print("1. INCREASE `TRT_WORKSPACE_GB` if you have more VRAM available and the OOM is during tactic selection.")
        print("2. DECREASE `TRT_OPT_BATCH_SIZE` (e.g., to 1 or a very small number).")
        print("3. Ensure no other processes are using significant VRAM (check nvidia-smi).")
        print("4. Use a machine with more VRAM for the build process.")
        print("5. If not using FP16, enable it if platform supports fast FP16.")
        print("6. Check ONNX model compatibility with TensorRT (use `trtexec` for detailed layer info).")
        # exit() # Or raise an error
    else:
        with open(TRT_ENGINE_FILENAME, "wb") as f:
            f.write(serialized_engine)
        print(f"TensorRT engine built and saved to: {TRT_ENGINE_FILENAME}")

except Exception as e:
    import traceback
    print(traceback.format_exc())
    print(f"AN UNEXPECTED EXCEPTION OCCURRED DURING ENGINE BUILDING: {e}")
    print("This could be an Out-Of-Memory (OOM) error or another configuration problem.")
    # exit() # Or raise an error
finally:
    # Cleanup
    del builder, network, config, parser, profile
    if 'serialized_engine' in locals() and serialized_engine is not None:
        del serialized_engine
    gc.collect()
    torch.cuda.empty_cache() # Good practice to free PyTorch CUDA cache if used prior
    print("Cleaned up TensorRT build objects.")


--- Step 2: Building TensorRT Engine ---
ONNX Input Model: yolov8m_bs1_img960x960_dynamic.onnx
Target Engine File: yolov8m_bs1-64-512_img960x960_fp16.engine
Min Batch Size: 1
Opt Batch Size: 1
Max Batch Size: 256
Image Size: 960x960
Precision: FP16
Workspace: 12 GB
This may take a VERY long time and require substantial VRAM...
FP16 mode enabled.
ONNX model parsed successfully.
Network Input Tensor Name used for profile: images
Optimization profile added for input 'images':
  Min Shape: (1, 3, 960, 960)
  Opt Shape: (1, 3, 960, 960)
  Max Shape: (256, 3, 960, 960)
Building serialized network (engine)...
ERROR: Failed to build the TensorRT engine. This is often due to OOM or configuration issues.
Try the following:
1. INCREASE `TRT_WORKSPACE_GB` if you have more VRAM available and the OOM is during tactic selection.
2. DECREASE `TRT_OPT_BATCH_SIZE` (e.g., to 1 or a very small number).
3. Ensure no other processes are using significant VRAM (check nvidia-smi).
4. Use a machine with more 

In [ ]:
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# Cell 6: (Optional) Test TensorRT Engine
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
print(f"\n--- Optional Step: Testing TensorRT Engine ({TRT_ENGINE_FILENAME}) ---")

if not os.path.exists(TRT_ENGINE_FILENAME):
    print(f"TensorRT engine {TRT_ENGINE_FILENAME} not found. Skipping test.")
else:
    # Import pycuda.driver and autoinit (MUST be done before any other PyCUDA calls)
    # autoinit ensures a CUDA context is created and available for PyCUDA.
    import pycuda.driver as cuda
    import pycuda.autoinit # IMPORTANT!

    runtime = trt.Runtime(TRT_LOGGER)
    with open(TRT_ENGINE_FILENAME, "rb") as f:
        engine = runtime.deserialize_cuda_engine(f.read())

    if not engine:
        print("ERROR: Failed to deserialize the TensorRT engine.")
        # exit() # Or raise an error
    else:
        print("TensorRT engine deserialized successfully.")
        context = engine.create_execution_context()

        if not context:
            print("ERROR: Failed to create execution context.")
            # exit() # Or raise an error
        else:
            # Determine input and output binding names and their order
            input_bindings_info = [] # List of (name, index)
            output_bindings_info = [] # List of (name, index)

            for i in range(engine.num_bindings):
                binding_name = engine.get_binding_name(i)
                if engine.binding_is_input(i):
                    input_bindings_info.append({'name': binding_name, 'index': i})
                    print(f"  Detected Input: {binding_name} at index {i}")
                else:
                    output_bindings_info.append({'name': binding_name, 'index': i})
                    print(f"  Detected Output: {binding_name} at index {i}")
            
            if not input_bindings_info:
                print("ERROR: No input bindings found in the engine.")
                # exit() # Or raise an error

            # Assuming the first detected input is the primary image input for this example
            primary_input_name = input_bindings_info[0]['name']

            # Test with a few batch sizes within the profile's range.
            test_batch_sizes = [TRT_MIN_BATCH_SIZE, TRT_OPT_BATCH_SIZE, TARGET_MAX_INFERENCE_BATCH_SIZE]
            # Filter out duplicates and sort
            test_batch_sizes = sorted(list(set(bs for bs in test_batch_sizes if bs <= TARGET_MAX_INFERENCE_BATCH_SIZE)))
            # For very large TARGET_MAX_INFERENCE_BATCH_SIZE, you might only test a subset for speed/VRAM.
            # e.g., test_batch_sizes = [TRT_MIN_BATCH_SIZE, min(TRT_OPT_BATCH_SIZE, 32)]


            for current_test_batch_size in test_batch_sizes:
                print(f"\nTesting inference with batch_size = {current_test_batch_size}")

                # Set input shape for the primary input tensor for the current batch size
                context.set_input_shape(primary_input_name, (current_test_batch_size, CHANNELS, IMAGE_SIZE_H, IMAGE_SIZE_W))
                # If other inputs have dynamic shapes, set them here too:
                # context.set_input_shape("other_input_name", (current_test_batch_size, ...other_dims))

                # Allocate host and device buffers
                bindings_mem = [] # Stores device pointers (as integers for execute_v2)
                host_inputs = {}  # Dict to store host input buffers {name: buffer}
                device_inputs = {} # Dict to store device input buffers {name: buffer}
                host_outputs = {} # Dict to store host output buffers {name: buffer}
                device_outputs = {}# Dict to store device output buffers {name: buffer}
                output_shapes_from_context = {} # Dict to store actual output shapes {name: shape}

                for binding_info in input_bindings_info + output_bindings_info:
                    binding_name = binding_info['name']
                    binding_idx = binding_info['index'] # Get the original index for context calls

                    # Get actual shape from context AFTER set_input_shape
                    actual_binding_shape = context.get_binding_shape(binding_idx)
                    binding_dtype = trt.nptype(engine.get_binding_dtype(binding_idx))
                    
                    if -1 in actual_binding_shape:
                        print(f"ERROR: Binding '{binding_name}' shape {actual_binding_shape} still has dynamic dimension after set_input_shape. Check profile/logic.")
                        continue # Skip this binding if shape is not resolved

                    volume = trt.volume(actual_binding_shape)
                    if volume == 0: # Can happen if a dimension is 0
                        print(f"Warning: Binding '{binding_name}' has zero volume with shape {actual_binding_shape}. Skipping memory allocation for it if not an error.")
                        # Decide how to handle zero-volume bindings; often indicates an issue or an optional output not present
                        bindings_mem.append(0) # Append a null pointer if this binding will not be used.
                                               # This assumes execute_v2 handles nullptrs for zero-volume correctly.
                                               # Or, error out if zero volume is unexpected.
                        continue


                    host_mem = cuda.pagelocked_empty(volume, dtype=binding_dtype)
                    device_mem = cuda.mem_alloc(host_mem.nbytes)
                    bindings_mem.append(int(device_mem)) # Add device pointer for execute_v2

                    if engine.binding_is_input(binding_idx):
                        # Create dummy input data
                        dummy_data = np.random.rand(*actual_binding_shape).astype(binding_dtype)
                        np.copyto(host_mem, dummy_data.ravel()) # ravel to ensure flat for copy
                        host_inputs[binding_name] = host_mem
                        device_inputs[binding_name] = device_mem
                    else:
                        host_outputs[binding_name] = host_mem
                        device_outputs[binding_name] = device_mem
                        output_shapes_from_context[binding_name] = actual_binding_shape

                # Transfer input data to the GPU.
                for name, h_input_mem in host_inputs.items():
                    cuda.memcpy_htod(device_inputs[name], h_input_mem)

                # Run inference.
                # execute_v2 uses a list of device pointers (bindings_mem needs to be in correct binding order)
                # Ensure bindings_mem is ordered by binding_idx if you constructed it out of order
                ordered_bindings_mem = [0] * engine.num_bindings
                for i in range(engine.num_bindings):
                    b_name = engine.get_binding_name(i)
                    if engine.binding_is_input(i) and b_name in device_inputs:
                        ordered_bindings_mem[i] = int(device_inputs[b_name])
                    elif not engine.binding_is_input(i) and b_name in device_outputs:
                         ordered_bindings_mem[i] = int(device_outputs[b_name])
                    # Handle cases where a binding might not have mem allocated (e.g. zero volume) if applicable

                context.execute_v2(bindings=ordered_bindings_mem)

                # Transfer predictions back from the GPU.
                for name, h_output_mem in host_outputs.items():
                    cuda.memcpy_dtoh(h_output_mem, device_outputs[name])

                # Process outputs (reshape them)
                print("Output details:")
                for name, h_output_mem in host_outputs.items():
                    shape = output_shapes_from_context[name]
                    print(f"  Output binding '{name}' (flat from host): {h_output_mem.shape}")
                    reshaped_output = h_output_mem.reshape(shape)
                    print(f"  Reshaped output '{name}': {reshaped_output.shape}")

                    if reshaped_output.shape[0] == current_test_batch_size:
                        print(f"    SUCCESS: Output batch size ({reshaped_output.shape[0]}) matches test batch size ({current_test_batch_size}).")
                    else:
                        # This check is only valid if the first dimension is indeed batch size
                        print(f"    WARNING: Output batch size ({reshaped_output.shape[0]}) does NOT match test batch ({current_test_batch_size}).")

                # Clean up memory for this batch iteration (optional if VRAM is plentiful, but good practice)
                # for mem in device_inputs.values(): mem.free()
                # for mem in device_outputs.values(): mem.free()
                # del host_inputs, device_inputs, host_outputs, device_outputs, bindings_mem, output_shapes_from_context, ordered_bindings_mem
                # gc.collect() # Might be too frequent here, but ensures cleanup

            # Clean up context and engine after all test batches
            del context
        del engine
    del runtime
    gc.collect()
    # PyCUDA context is auto-cleaned by autoinit on script exit, but explicit cleanup of allocations is good.
    # Note: PyCUDA memory (device_mem) allocated with cuda.mem_alloc() is automatically freed when
    # the 'device_mem' object goes out of scope IF PyCUDA's context management is active.
    # However, explicitly calling .free() on device_mem objects before they go out of scope
    # can be done if you need finer control, especially in loops.
    # For this script structure, PyCUDA's auto-cleanup is often sufficient.
    torch.cuda.empty_cache() # If PyTorch was used and might hold onto cache
    print("TensorRT engine test complete and objects cleaned up.")